# 01 — Overview: who responded and what goes wrong

Response coverage, the problems groups report (and on which systems), and the mix of workloads.

*Principles: P1, P12 in the [analysis framework](https://github.com/svaradh/hpc-questionnaire/blob/main/docs/analysis-framework.md). Nothing here scores or ranks groups; flags are candidates for the committee to look at. Totals cover responding groups only, and "Don't know" is never counted as zero; a yes/no flag shown as `<NA>` means its inputs are unknown. Only the respondents' answers and the cluster configuration are used — nothing depends on which code a workload runs.*

**To run:** paste the Spreadsheet ID in the first code cell, then *Runtime → Run all*. Colab asks once per session to sign in with your IISER Google account.

In [ ]:
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'   # <- the long string in the sheet URL
REPO_REF = 'main'                                   # git branch/tag for the helper code and cluster config

import os, urllib.request
if not os.path.exists('hpc_analysis.py'):            # in Colab: fetch the helper modules once per session
    for f in ['sheets_client.py', 'hpc_analysis.py', 'hpc_plots.py', 'code_capabilities.json']:
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/svaradh/hpc-questionnaire/{REPO_REF}/analysis/{f}', f)

import pandas as pd
from sheets_client import load_data, load_config
import hpc_analysis as ha
import hpc_plots as hp
pd.set_option('display.max_colwidth', 120)

dfs = load_data(SPREADSHEET_ID)
cfg = load_config(ref=REPO_REF)
wl = ha.workloads(dfs, cfg)
cov = ha.coverage(dfs)
print(f"{cov['groups']} groups responded; {cov['workloads']} workloads described "
      f"({cov['groups_with_workloads']} groups with at least one).")

## Problems reported

The pie shows how the problem *reports* divide among problem types (a group can report several).
The bar chart below it gives the share of *groups* reporting each problem, which is the figure to quote.

In [ ]:
pr = ha.problems(dfs)
issues = pr[~pr['problem'].isin(['no_problems_observed', 'no_hpc_experience', 'dont_know'])]
hp.pie(issues['label'], issues['groups'], 'Problem reports by type')
hp.barh(pr['label'], pr['share_of_groups'], 'Share of groups reporting each problem', pct=True)
pr

## Problems faced on Gargi and on Raman

Each problem reported above was asked about again for Gargi and for Raman separately (their hardware and policies
differ; Gargi is newer and its policies are still settling). `elsewhere_only` = reported, but faced on neither — context
from other systems. In the cluster table, `work_kept_off_some` / `work_kept_off_all` count groups whose problems on that
cluster kept some, or all, of their work off it (run elsewhere or not at all): demand displaced by limits or policy.

In [ ]:
on = pr[pr['on_gargi'] + pr['on_raman'] > 0]
if len(on):
    hp.paired(on, 'label', 'on_gargi', 'on_raman', 'Gargi', 'Raman', 'Groups facing each problem, by cluster', 'Groups')
display(ha.cluster_engagement(dfs))
pr[['label', 'groups', 'on_gargi', 'on_raman', 'elsewhere_only']]

## Systems used and share of computing done at IISER Bhopal

In [ ]:
from sheets_client import explode_semicolons
su = explode_semicolons(dfs['Experience'], 'systems_used').value_counts()
hp.barh(su.index, su.values, 'Systems groups ran jobs on (last 3 years)', 'Groups', integer=True)
ha.num(dfs['Experience']['share_iiserb_pct']).describe()

## Workload mix

In [ ]:
ty = explode_semicolons(dfs['Workloads'], 'type').value_counts()
hp.barh(ty.index, ty.values, 'Workloads by type of computation', 'Workloads', integer=True)
display(wl.groupby(['device', 'cluster']).size().rename('workloads').reset_index())
hp.counts(wl['evidence_level'].map(lambda v: f'Level {v:.0f}' if v == v else 'unknown'), 'Evidence level of workload numbers (1 = direct observation)')

## Group size and growth

In [ ]:
g = dfs['Group'].assign(now=lambda d: ha.num(d['hpc_users_now']), in_3y=lambda d: ha.num(d['hpc_users_3y']))
print(f"Users now: {g['now'].sum():.0f} ({g['now'].isna().sum()} unknown); in 3 years: {g['in_3y'].sum():.0f} ({g['in_3y'].isna().sum()} unknown)")
ha.num(dfs['Closing']['growth_3y_factor']).describe()